In [1]:
# =====================================================
# MÓDULO 2
# Evaluación del Estilo de Vida
# Dataset: Sleep Health and Lifestyle
# Algoritmos:
#   - Decision Tree
#   - Random Forest
#
# Fix aplicado (revisión backend, 2026-07-25):
#   La version anterior usaba dropna(subset=["Sleep Disorder"]), lo
#   que descartaba 219 de 374 filas (las personas SIN trastorno) y
#   convertia el problema en clasificar Insomnia vs Sleep Apnea
#   unicamente. Eso no es lo que pide la Fase 4 del documento
#   (categoria de estilo de vida Excelente/Bueno/Regular/Malo) y
#   ademas tira la mayoria del dataset.
#
#   Cambios:
#   1) Sleep Disorder se rellena como "None" en vez de eliminarse
#      -> se mantienen las 374 filas y un problema de 3 clases real
#      (None / Insomnia / Sleep Apnea). Este SI es un target legitimo
#      para ML: no se puede derivar con una formula a partir de las
#      demas columnas, es una condicion medica real reportada.
#   2) La categoria "Excelente/Bueno/Regular/Malo" que pide el
#      documento se calcula aparte con una formula simple (no ML,
#      igual que un cuestionario tipo FINDRISC) a partir de horas de
#      sueño, calidad, actividad fisica y estres. Se deja aqui solo
#      a modo de referencia/EDA; el backend puede recalcularla
#      directamente sin necesidad de cargar un modelo.
#   3) Se excluyen "BMI Category" y "Blood Pressure" de X: no estan
#      en la lista de variables que la Fase 4 dice que la app le
#      pedira al usuario para este modulo, y pedirlas duplicaria
#      datos que ya se piden distinto para el Modulo 1 (peso/estatura,
#      HighBP). El modelo debe entrenarse solo con lo que el
#      formulario realmente va a enviar.
# =====================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# =========================
# Cargar dataset
# =========================

df = pd.read_csv("../data/Sleep_health_and_lifestyle_dataset.csv")
df = df.drop_duplicates()

if "Person ID" in df.columns:
    df = df.drop(columns=["Person ID"])

print(df.head())
print(df.info())
print(df.isnull().sum())

# =========================
# Categoría de estilo de vida (regla, no ML) - referencia Fase 4
# =========================

def lifestyle_category(row):
    score = (
        min(row["Sleep Duration"] / 9, 1) * 0.25
        + (row["Quality of Sleep"] / 10) * 0.25
        + (row["Physical Activity Level"] / 100) * 0.25
        + ((10 - row["Stress Level"]) / 10) * 0.25
    )
    if score >= 0.75:
        return "Excelente"
    elif score >= 0.55:
        return "Bueno"
    elif score >= 0.35:
        return "Regular"
    else:
        return "Malo"

df["Lifestyle_Category"] = df.apply(lifestyle_category, axis=1)
print("\nDistribución de Lifestyle_Category (calculada por fórmula, no ML)")
print(df["Lifestyle_Category"].value_counts())

# =========================
# Target ML real: Sleep Disorder (None / Insomnia / Sleep Apnea)
# =========================

df["Sleep Disorder"] = df["Sleep Disorder"].fillna("None")

X = df.drop(columns=["Sleep Disorder", "Lifestyle_Category", "BMI Category", "Blood Pressure"])
y = df["Sleep Disorder"]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# =====================================================
# MODELO 1 - DECISION TREE
# =====================================================

modelo_dt = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("modelo", DecisionTreeClassifier(random_state=42, class_weight="balanced"))
])
modelo_dt.fit(X_train, y_train)
pred_dt = modelo_dt.predict(X_test)

print("\n===============================")
print("DECISION TREE")
print("===============================")
print("Accuracy :", round(accuracy_score(y_test, pred_dt), 4))
print("Precision:", round(precision_score(y_test, pred_dt, average="weighted"), 4))
print("Recall   :", round(recall_score(y_test, pred_dt, average="weighted"), 4))
print("F1 Score :", round(f1_score(y_test, pred_dt, average="weighted"), 4))
print("\nMatriz de Confusión")
print(confusion_matrix(y_test, pred_dt))
print("\nClassification Report")
print(classification_report(y_test, pred_dt))

# =====================================================
# MODELO 2 - RANDOM FOREST
# =====================================================

modelo_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("modelo", RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight="balanced"
    ))
])
modelo_rf.fit(X_train, y_train)
pred_rf = modelo_rf.predict(X_test)

print("\n===============================")
print("RANDOM FOREST")
print("===============================")
print("Accuracy :", round(accuracy_score(y_test, pred_rf), 4))
print("Precision:", round(precision_score(y_test, pred_rf, average="weighted"), 4))
print("Recall   :", round(recall_score(y_test, pred_rf, average="weighted"), 4))
print("F1 Score :", round(f1_score(y_test, pred_rf, average="weighted"), 4))
print("\nMatriz de Confusión")
print(confusion_matrix(y_test, pred_rf))
print("\nClassification Report")
print(classification_report(y_test, pred_rf))

# =====================================================
# COMPARACIÓN FINAL Y SELECCIÓN DEL GANADOR
# =====================================================

resultados = pd.DataFrame({
    "Modelo": ["Decision Tree", "Random Forest"],
    "Accuracy": [accuracy_score(y_test, pred_dt), accuracy_score(y_test, pred_rf)],
    "Precision": [
        precision_score(y_test, pred_dt, average="weighted"),
        precision_score(y_test, pred_rf, average="weighted"),
    ],
    "Recall": [
        recall_score(y_test, pred_dt, average="weighted"),
        recall_score(y_test, pred_rf, average="weighted"),
    ],
    "F1": [
        f1_score(y_test, pred_dt, average="weighted"),
        f1_score(y_test, pred_rf, average="weighted"),
    ],
})

print("\n===============================")
print("COMPARACIÓN DE MODELOS")
print("===============================")
print(resultados)

candidatos = {"Decision Tree": modelo_dt, "Random Forest": modelo_rf}
ganador_nombre = resultados.sort_values("Recall", ascending=False).iloc[0]["Modelo"]
ganador_pipeline = candidatos[ganador_nombre]

print(f"\nModelo ganador por Recall: {ganador_nombre}")

joblib.dump(ganador_pipeline, "../models_artifacts/modelo2_estilo_vida.joblib")
print("Guardado en ../models_artifacts/modelo2_estilo_vida.joblib")
print("Columnas esperadas por el modelo:", list(X.columns))


  Gender  Age            Occupation  Sleep Duration  Quality of Sleep  \
0   Male   27     Software Engineer             6.1                 6   
1   Male   28                Doctor             6.2                 6   
2   Male   28                Doctor             6.2                 6   
3   Male   28  Sales Representative             5.9                 4   
4   Male   28  Sales Representative             5.9                 4   

   Physical Activity Level  Stress Level BMI Category Blood Pressure  \
0                       42             6   Overweight         126/83   
1                       60             8       Normal         125/80   
2                       60             8       Normal         125/80   
3                       30             8        Obese         140/90   
4                       30             8        Obese         140/90   

   Heart Rate  Daily Steps Sleep Disorder  
0          77         4200            NaN  
1          75        10000            Na

/tmp/ipykernel_142466/1845192001.py:109: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns



RANDOM FOREST
Accuracy : 0.8267
Precision: 0.8622
Recall   : 0.8267
F1 Score : 0.835

Matriz de Confusión
[[13  0  2]
 [ 3 36  5]
 [ 3  0 13]]

Classification Report
              precision    recall  f1-score   support

    Insomnia       0.68      0.87      0.76        15
        None       1.00      0.82      0.90        44
 Sleep Apnea       0.65      0.81      0.72        16

    accuracy                           0.83        75
   macro avg       0.78      0.83      0.80        75
weighted avg       0.86      0.83      0.84        75


COMPARACIÓN DE MODELOS
          Modelo  Accuracy  Precision    Recall        F1
0  Decision Tree  0.826667   0.862175  0.826667  0.835015
1  Random Forest  0.826667   0.862175  0.826667  0.835015

Modelo ganador por Recall: Decision Tree
Guardado en ../models_artifacts/modelo2_estilo_vida.joblib
Columnas esperadas por el modelo: ['Gender', 'Age', 'Occupation', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 'Stress Level', 'Heart